In [2]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-01-22 15:05:55--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Résolution de raw.githubusercontent.com (raw.githubusercontent.com)… 2606:50c0:8001::154, 2606:50c0:8000::154, 2606:50c0:8003::154, ...
Connexion à raw.githubusercontent.com (raw.githubusercontent.com)|2606:50c0:8001::154|:443… connecté.
requête HTTP transmise, en attente de la réponse… 200 OK
Taille : 1115394 (1,1M) [text/plain]
Sauvegarde en : « input.txt.1 »

input.txt.1         100%[===================>]   1,06M  --.-KB/s    ds 0,07s   

2026-01-22 15:05:56 (14,8 MB/s) — « input.txt.1 » sauvegardé [1115394/1115394]



In [3]:
#paramètres 
import torch
batch_size=32 #nombre de séquences traitées en parallèle
block_size=8  #longueur de chaque séquence
max_iters=10000 #nombre d'itérations
eval_interval=500 #intervalle d'évaluation de la loss
learning_rate=1e-3 #taux d'apprentissage
device='cuda' if torch.cuda.is_available() else 'cpu' #utilisation du GPU 
eval_iters=200 #nombre d'itérations pour l'évaluation de la loss

In [4]:
with open('input.txt', 'r') as f:
    text = f.read()

In [5]:
print(f"Length of dataset in characters: {len(text)}")

Length of dataset in characters: 1115394


In [6]:
print(text[:100]) 

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [7]:
chars=sorted(list(set(text)))
print(type(chars))
vocab_size=len(chars)
print("Vocab size:",vocab_size, chars)

<class 'list'>
Vocab size: 65 ['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


Encodage: représentation sous forme de vecteur d'entier codé lettre par lettre 

In [8]:
stoi={ch:i for i,ch in enumerate(chars)}
itos={i:ch for i,ch in enumerate(chars)}
encode = lambda x:[stoi[i] for i in x ]
decode = lambda x: "".join ([itos[i] for i in x])

print(encode("Bonjour"))
print(decode(encode("Bonjour")))

[14, 53, 52, 48, 53, 59, 56]
Bonjour


Exemple avec un autre type d'encodage (par sous-mot)

In [9]:

import tiktoken
enc=tiktoken.get_encoding("gpt2")
print(enc.encode("Bonjour"))
print(enc.decode(enc.encode("Bonjour")))

[20682, 73, 454]
Bonjour


encodage complet des datas

In [10]:


import torch 
encoded_text=torch.tensor(encode(text),dtype=torch.long)
print(encoded_text.shape, encoded_text.dtype)
print(encoded_text[:100])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


Séparation en trainig data set et validation data set

In [11]:
n = int(0.9 * len(encoded_text))
train_data = encoded_text[:n]
val_data = encoded_text[n:]


On ne rentre jamais tout le texte d'un coup en entrée du "transformers" pour l'entrainer. On utilise plutot des morceaux de data d'une certaine longueur (avec un max défini appelé block_size)

In [12]:
block_size = 8
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

On rajoute un $+1$ car en réalité, dans un morceau de texte de longueur $\text{block\_size}$, il y a $\text{block\_size}-1$ exemples à analyser : 

* $47$ qui suit $18$
* $56$ qui suit $18, 47$
* ...

Il y a le contexte et la lettre à prédire qui vient après 

In [13]:
x = train_data[:block_size]
y = train_data[1:block_size+1] # la sortie est decalée d'un cran car on veut predire le caractere suivant
for t in range(block_size):
    context= x[:t+1]
    target = y[t]
    print (f"quand l'entrée est {context}, la sortie est {target} ")


quand l'entrée est tensor([18]), la sortie est 47 
quand l'entrée est tensor([18, 47]), la sortie est 56 
quand l'entrée est tensor([18, 47, 56]), la sortie est 57 
quand l'entrée est tensor([18, 47, 56, 57]), la sortie est 58 
quand l'entrée est tensor([18, 47, 56, 57, 58]), la sortie est 1 
quand l'entrée est tensor([18, 47, 56, 57, 58,  1]), la sortie est 15 
quand l'entrée est tensor([18, 47, 56, 57, 58,  1, 15]), la sortie est 47 
quand l'entrée est tensor([18, 47, 56, 57, 58,  1, 15, 47]), la sortie est 58 


On prend en contexte tous les chaines de caractère de longueure 1 à block_size pour habiter le transformers à toutes les longueures de contexte

On généralise l'exemple ci-dessus à tout le texte avec la notion de batch

In [14]:
torch.manual_seed(1337)
batch_size = 4
block_size = 8
def get_batch(selection):
    data = train_data if selection == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x,y = x.to(device), y.to(device)
    return x, y

#évaluation de la loss avec une moyenn sur eval_iters itérations pour avoir moins de bruit
@torch.no_grad #Tour ce qui se passe n'est pas mémorisé pour le calcul du gradient
def estimate_loss():
    out = {}
    model.eval() #mode évaluation
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train() #remettre le modèle en mode entrainement
    return out

xb, yb = get_batch('train')
print('entrée:')
print(xb)
print('sortie:')
print(yb)
print(yb.shape)



entrée:
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
sortie:
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
torch.Size([4, 8])


SImple NN pour commencer : Bigram Language Model

In [15]:
import torch.nn as nn
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        #On configure une table d'embedding qui va mapper chaque token de l'entrée à un vecteur de la taille du vocabulaire dont la valeur sera utilisée pour prédire le token suivant
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx,targets =None):
        #Idx est un tenseur d'indices de tokens de forme (batch_size, block_size)
        #On retourne les logits de forme (batch_size, block_size, vocab_size)
        logits = self.token_embedding_table(idx)
        #Chaque position dans la séquence d'entrée a maintenant un vecteur de logits qui prédit le token suivant
        if targets is None: #Si les targets ne sont pas fournies, on ne calcule pas la loss afin de pouvoir faire de la génération de texte
            loss = None
        else:
            #Calcul de la loss si les targets sont fournies
            B,T,C = logits.shape #Batch size, Time steps (ou block size), Channels (ou vocab size)

            logits = logits.view(B*T, C) #On aplati les dimensions batch et time pour calculer la loss plus facilement
            targets = targets.view(B*T)  #On fait de même pour les targets
            #Mesure de Loss

            loss = nn.functional.cross_entropy(logits, targets) #cross_entropy est la negative log likelihood loss

        return logits, loss
    
    #Generation de texte

    def generate(self, idx, max_new_tokens):
        #idx est un tenseur de forme (batch_size, sequence_length) qui est le contexte initial
        for _ in range(max_new_tokens):
            logits, loss = self(idx)  
            #On s'intéresse seulement au dernier pas de temps
            logits = logits[:, -1, :]  #On prend les logits du dernier token de la séquence

            probs = nn.functional.softmax(logits, dim=-1)  #On convertit les logits en probabilités avec softmax

            #On échantillonne le prochain token à partir de la distribution de probabilité

            next_token = torch.multinomial(probs, num_samples=1)  #On échantillonne un token

            #On ajoute le token échantillonné à la séquence d'entrée

            idx = torch.cat((idx, next_token), dim=1)  #On concatène le nouveau token à la séquence existante
        return idx

model = BigramLanguageModel(vocab_size)
model = model.to(device)
logits, loss = model(xb, yb)
print("logits shape:", logits.shape)
print("loss:", loss)

idx=torch.zeros((1,1),dtype=torch.long)
print(decode(model.generate(idx, max_new_tokens=100)[0].tolist())) #completement aléatoire car non entrainé 

logits shape: torch.Size([32, 65])
loss: tensor(4.8786, grad_fn=<NllLossBackward0>)

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


On a un résulats aléatoir car on a pas entrainé le model

In [16]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

In [17]:
batch_size = 32
for steps in range(max_iters):
    
    if steps % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {steps}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch('train')    
    logits, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True) #on remet les gradients à zéro avant la backward pass

    loss.backward() #calcul des gradients

    optimizer.step()#mise à jour des poids

print(loss.item())


step 0: train loss 4.7296, val loss 4.7235
step 500: train loss 4.1778, val loss 4.1749
step 1000: train loss 3.7325, val loss 3.7325
step 1500: train loss 3.3875, val loss 3.3882
step 2000: train loss 3.1145, val loss 3.1367
step 2500: train loss 2.9349, val loss 2.9346
step 3000: train loss 2.7901, val loss 2.8084
step 3500: train loss 2.7045, val loss 2.7173
step 4000: train loss 2.6474, val loss 2.6427
step 4500: train loss 2.5931, val loss 2.6018
step 5000: train loss 2.5614, val loss 2.5746
step 5500: train loss 2.5415, val loss 2.5420
step 6000: train loss 2.5148, val loss 2.5286
step 6500: train loss 2.4982, val loss 2.5127
step 7000: train loss 2.4854, val loss 2.5082
step 7500: train loss 2.4888, val loss 2.4977
step 8000: train loss 2.4785, val loss 2.4903
step 8500: train loss 2.4789, val loss 2.4865
step 9000: train loss 2.4620, val loss 2.4891
step 9500: train loss 2.4681, val loss 2.4909
2.499467611312866


In [18]:
idx=torch.zeros((1,1),dtype=torch.long,device=device)
print(decode(model.generate(idx,max_new_tokens=1000)[0].tolist())) #texte généré après entrainement


Tordillape ngt VI f helin:
ENIf w t be moupo arais GBead pripi.
SCh'lice'd ld; w; haur ingus libhat utamiannot bendousid, woucore ISTEO, l ber bors r, at t,
rt tesaireny.
MI yowanckin ove bo-oulat.
I' por atBulay he Ir por borille:
ghs MABr as dary wnay, ans my we tel:
V:
The,

LUCELyZAn, chthow gre irne higedr woff hen's hory herarames ve s yo juppas Her ly wh anthe icemerat anghucher thesintes bongimentourid grturde IAr ate; whemy woreers.
PUnjRCl.y ndlen IUCHUELZYO,
Wiced d mow;
Hos am:
LiCE:
ULONCo aticlle lly fatheghen inello tmeouthinont. coron.
3$g; mbebod nt
To e G mLIVeser asecice moun'Taloofane dililf Ithith hiting an:
Win:
KELie hff kicerowhothetomachit ofourseiur iof herelin bephellalyoopry bef wn Ther, pratil,
NEDUS:
Horatiss bugon t hele zernce cuind 'Theshelllld tor t mailinof:
Tonathe taliototheand onougbe?
wices pin thay, ar ve HAButhaithar?

To bese ou, gu n Y thesthed:

Dist ighariso anely, t ENUSICo ay hy,
Twiviourive: shay mies neriles! zedolo talidrgollsthe.
ARKm

Il n y a pas de cohérence car les tokens ne "communiquent pas entre eux" on regarde juste le dernier character pour prédire le suivant.

Self-attention

In [19]:
torch.manual_seed(1337)
B,T,C=4,8,2
x=torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

On veut avoir en contexte tout les information AVANT le toke qui est en train d'être traité.
On fait une moyenne de tous les token précédent

In [21]:
#On veut x[b,t] qui est la moyenne (temporelle) de x[b, :t+1]
xbow=torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xbow[b,t]=x[b,:t+1].mean(0)


In [26]:
xbow[0]


tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])

In [25]:
x[0]

tensor([[ 0.1808, -0.0700],
        [-0.3596, -0.9152],
        [ 0.6258,  0.0255],
        [ 0.9545,  0.0643],
        [ 0.3612,  1.1679],
        [-1.3499, -0.5102],
        [ 0.2360, -0.2398],
        [-0.9211,  1.5433]])

Le résultat ci-dessus n'est pas du tout efficace. On peut amlirer cela avec le calcul matriciel (et une matrice triangulaire inférieur qui va permettre une considération "temporelle").



In [ ]:
wei=torch.tril(torch.ones(T,T)) #T est le block size (la longueur de la séquence)
wei=wei/wei.sum(1,keepdim=True) #On obtient un matrice de moyenne cumulative

xbow2=wei @ x #(T,T) @ (B,T,C) -> (BT,C)

torch.allclose(xbow, xbow2) #on vérifie que les deux méthodes donnent le même résultat

True

Dernière Version de l'écrire (même résultat) utilisant softmax

In [30]:
trill=torch.tril(torch.ones(T,T))
wei=torch.zeros(T,T)
wei=wei.masked_fill(trill==0, float('-inf'))
wei=wei.softmax(1)
print(wei)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])


In [31]:
xbow3=wei @ x
torch.allclose(xbow3, xbow)

True

Version 4 : Self attention

In [ ]:
torch.manual_seed(1337)
B,T,C=4,8,32
x=torch.randn(B,T,C)

trill=torch.tril(torch.ones(T,T))
wei=torch.zeros(T,T) #On ne veut pas forcement que tous les tokens regardent tous les autres tokens de la même manière (certains sont pus importants que d'autres)
wei=wei.masked_fill(trill==0, float('-inf'))
wei=wei.softmax(1)
xbow=wei @ x
xbow.shape

torch.Size([4, 8, 32])

Tous les token vont emettrent 2 infos : 
* Une requette (qu'est ce que le token "cherche")
* Une clef (qu'est ce que le token contient)

On obtient les "affinité" entre des token en faisant le produit scalaire entre la requette d'un token et la clef d'un autre

Cette "affinité" devient le "poid" du token dans la moyenne que nous avons fais précédement 

In [ ]:
torch.manual_seed(1337)
B,T,C=4,8,32
x=torch.randn(B,T,C)

#On utilise un "Head" d'attention

head_size=16

clef=nn.Linear(C, head_size, bias=False)
requette=nn.Linear(C, head_size, bias=False)
k=clef(x) # (B,T,head_size)
q=requette(x) #(B,T,head_size)

value=nn.Linear(C, head_size, bias=False)

wei=q @ k.transpose(-2,-1)  #(B,T,head_size) @ (B, head_size,T) -> (B,T,T). Les éléments dans les différents batches ne communique pas entre eux 

trill=torch.tril(torch.ones(T,T))
wei=wei.masked_fill(trill==0, float('-inf'))
wei=torch.softmax(wei, dim=-1)
#xbow=wei @ x

v=value(x)

#on n'utilise plus x mais v car c'est lavaleur associée à chaque token
xbow=wei @ v
print(wei)

tensor([[[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1574, 0.8426, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.2088, 0.1646, 0.6266, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.5792, 0.1187, 0.1889, 0.1131, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0294, 0.1052, 0.0469, 0.0276, 0.7909, 0.0000, 0.0000, 0.0000],
         [0.0176, 0.2689, 0.0215, 0.0089, 0.6812, 0.0019, 0.0000, 0.0000],
         [0.1691, 0.4066, 0.0438, 0.0416, 0.1048, 0.2012, 0.0329, 0.0000],
         [0.0210, 0.0843, 0.0555, 0.2297, 0.0573, 0.0709, 0.2423, 0.2391]],

        [[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1687, 0.8313, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.2477, 0.0514, 0.7008, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.4410, 0.0957, 0.3747, 0.0887, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0069, 0.0456, 0.0300, 0.7748, 0.1427, 0.0000, 0.0000, 0.0000],
         [0.0660, 0.089

On parle de self attention parce que la clef, la requette et la valeur sont généré par x.
On pourrait immaginer un méchanisme d'attention dans lequel la clef serait généré par un contexte imposé d'une aures source

Dans l'article Attention is all you need, la formule complete du module d'attention est : 

$$SelfAttention =Softmax(\frac{QV^T}{d_k ^{1/2}})V$$

l'interet de divisé par la racine du head_size est de conserver la variance des poids

In [40]:
k=torch.randn(B,T,head_size)
q=torch.randn(B,T,head_size)
wei=q@k.transpose(-2,-1)

In [41]:
k.var()

tensor(0.9487)

In [42]:
q.var()

tensor(1.0449)

In [43]:
wei.var()

tensor(14.3682)

la variance de la matrice des poids est beacoup plus élevé que celle des matrice de clef et de requette, et cela est amplifié lorsqu'on applique softmax

In [45]:
torch.softmax(torch.tensor([0.1, 0.2, 0.3,-0.2]),dim=-1)

tensor([0.2459, 0.2717, 0.3003, 0.1821])

In [47]:
torch.softmax(torch.tensor([0.1, 0.2, 0.3,-0.2])*10,dim=-1)

tensor([0.0896, 0.2436, 0.6623, 0.0045])

Les poids vont surtout s'aggrégé autour de la plus grande valeure

In [48]:
torch.manual_seed(1337)
B,T,C=4,8,32
x=torch.randn(B,T,C)

#On utilise un "Head" d'attention

head_size=16

clef=nn.Linear(C, head_size, bias=False)
requette=nn.Linear(C, head_size, bias=False)
k=clef(x) # (B,T,head_size)
q=requette(x) #(B,T,head_size)

value=nn.Linear(C, head_size, bias=False)

wei=q @ k.transpose(-2,-1)*(head_size)**-0.5  #(B,T,head_size) @ (B, head_size,T) -> (B,T,T). Les éléments dans les différents batches ne communique pas entre eux 

trill=torch.tril(torch.ones(T,T))
wei=wei.masked_fill(trill==0, float('-inf'))
wei=torch.softmax(wei, dim=-1)
#xbow=wei @ x

v=value(x)

#on n'utilise plus x mais v car c'est lavaleur associée à chaque token
xbow=wei @ v
print(wei)

tensor([[[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.3966, 0.6034, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.3069, 0.2892, 0.4039, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.3233, 0.2175, 0.2443, 0.2149, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1479, 0.2034, 0.1663, 0.1455, 0.3369, 0.0000, 0.0000, 0.0000],
         [0.1259, 0.2490, 0.1324, 0.1062, 0.3141, 0.0724, 0.0000, 0.0000],
         [0.1598, 0.1990, 0.1140, 0.1125, 0.1418, 0.1669, 0.1061, 0.0000],
         [0.0845, 0.1197, 0.1078, 0.1537, 0.1086, 0.1146, 0.1558, 0.1553]],

        [[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.4016, 0.5984, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.3365, 0.2271, 0.4364, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.3019, 0.2060, 0.2899, 0.2022, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1058, 0.1700, 0.1530, 0.3451, 0.2261, 0.0000, 0.0000, 0.0000],
         [0.1526, 0.164